# LLM annotation through an API, one line at a time

Session 3, live demo. Run the cells **one by one**, top to bottom. After the setup cell, almost every cell is one line.

**Before you start:** in the left sidebar of Colab, click the key icon (Secrets), add `ANTHROPIC_API_KEY`, paste your key as the value, and switch on "Notebook access". Never paste a key into a cell.

The sentences are 24 invented sentences in the style of French manifestos, with the domain I would give each (`my_label`). They are not Manifesto Project data.

## 0. Setup (run once; nothing to explain here)

In [ ]:
%pip install -q anthropic pandas pydantic

import hashlib, io, json, os, time
from datetime import date
from typing import Literal

import anthropic, pandas as pd, requests
from pydantic import BaseModel

try:
    from google.colab import userdata
    KEY = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    KEY = os.environ.get("ANTHROPIC_API_KEY")
assert KEY, "No key: add ANTHROPIC_API_KEY in Colab Secrets (key icon, left sidebar), with Notebook access on."

MODEL = "claude-haiku-5-5"
URL = "https://api.anthropic.com/v1/messages"
HEADERS = {"x-api-key": KEY, "anthropic-version": "2023-06-01", "content-type": "application/json"}

CODEBOOK = """You code quasi-sentences of French party manifestos into one policy domain.
- external_relations: other countries, the military, peace, the European Union.
- freedom_democracy: freedom and human rights, democracy, constitutionalism.
- political_system: decentralisation, administrative efficiency, corruption, political authority.
- economy: markets, regulation, planning, protectionism, growth, infrastructure, budget discipline.
- welfare_quality_of_life: environment, culture, equality, welfare state, education.
- fabric_of_society: national way of life, traditional morality, law and order, multiculturalism.
- social_groups: workers, farmers, middle class, minorities, women, the elderly, the young.
- no_category: no policy content.
Answer with a one-sentence reason in English, then the domain."""

DOMAINS = ["external_relations", "freedom_democracy", "political_system", "economy",
           "welfare_quality_of_life", "fabric_of_society", "social_groups", "no_category"]


class Annotation(BaseModel):
    reason: str
    domain: Literal["external_relations", "freedom_democracy", "political_system", "economy",
                    "welfare_quality_of_life", "fabric_of_society", "social_groups", "no_category"]


SCHEMA = Annotation.model_json_schema() | {"additionalProperties": False}

SENTENCES = pd.read_csv(io.StringIO("""text,my_label
Nous sortirons du commandement intégré de l'OTAN.,external_relations
La France doit retrouver sa voix au Conseil de sécurité des Nations unies.,external_relations
Nous renégocierons les traités européens pour rendre aux Français leur souveraineté.,external_relations
Le référendum d'initiative citoyenne sera inscrit dans la Constitution.,freedom_democracy
La liberté de la presse et le secret des sources seront garantis.,freedom_democracy
Nous réduirons de moitié le nombre de parlementaires.,political_system
Les régions recevront de nouvelles compétences en matière de transport et de formation.,political_system
Aucun ministre mis en examen ne restera au gouvernement.,political_system
Nous baisserons les impôts de production pour relocaliser l'industrie.,economy
Le déficit public sera ramené sous les 3 % du PIB avant la fin du quinquennat.,economy
Les autoroutes seront renationalisées.,economy
Le très haut débit sera disponible dans chaque commune rurale.,economy
Le SMIC sera porté à 1 600 euros net.,welfare_quality_of_life
Nous recruterons 20 000 enseignants et limiterons les classes à 24 élèves.,welfare_quality_of_life
La France sortira du charbon et investira massivement dans la rénovation thermique.,welfare_quality_of_life
Aucun hôpital de proximité ne sera fermé.,welfare_quality_of_life
Les peines planchers seront rétablies pour les récidivistes.,fabric_of_society
Nous recruterons 10 000 policiers et gendarmes supplémentaires.,fabric_of_society
La laïcité est le ciment de la République et ne se négocie pas.,fabric_of_society
Les agriculteurs doivent pouvoir vivre de leur travail grâce à des prix rémunérateurs.,social_groups
L'égalité salariale entre les femmes et les hommes sera contrôlée dans chaque entreprise.,social_groups
Les petites retraites des artisans et commerçants seront revalorisées.,social_groups
"Françaises et Français, l'heure est venue de choisir.",no_category
Ensemble nous ferons de la France un pays fier et rassemblé.,no_category
"""))


def params(text):
    # The body of one request, as the API wants it. Used for one call and for a batch.
    return {"model": MODEL, "max_tokens": 300, "system": CODEBOOK, "thinking": {"type": "disabled"},
            "messages": [{"role": "user", "content": text}],
            "output_config": {"format": {"type": "json_schema", "schema": SCHEMA}}}


def text_of(message):
    return next(b.text for b in message.content if b.type == "text")


client = anthropic.Anthropic(api_key=KEY, max_retries=5)


def label(text):
    return json.loads(text_of(client.messages.create(**params(text))))["domain"]


print(f"{len(SENTENCES)} sentences, model {MODEL}, key found")

## 1. An API call is an HTTP request

No library, no magic: an address, headers, a JSON body. The key goes in a header.

In [ ]:
{**HEADERS, "x-api-key": "sk-ant-…hidden"}

In [ ]:
BODY = {"model": MODEL, "max_tokens": 300, "system": CODEBOOK, "thinking": {"type": "disabled"}, "messages": [{"role": "user", "content": SENTENCES.text[0]}]}

In [ ]:
SENTENCES.text[0]

In [ ]:
r = requests.post(URL, headers=HEADERS, json=BODY)

In [ ]:
r.status_code

In [ ]:
r.json()

The answer is JSON. The text is inside `content`; the bill is in `usage`.

In [ ]:
r.json()["content"][0]["text"]

In [ ]:
r.json()["usage"]

## 2. Ask for a label, not a paragraph

We give the API a JSON schema: a `reason`, then a `domain` from the list. The model cannot answer outside it.

In [ ]:
SCHEMA

In [ ]:
BODY["output_config"] = {"format": {"type": "json_schema", "schema": SCHEMA}}

In [ ]:
r = requests.post(URL, headers=HEADERS, json=BODY)

In [ ]:
json.loads(r.json()["content"][0]["text"])

## 3. The Python library does the same thing

`client.messages.create` builds that HTTP request for us. `parse` also checks the answer against the schema.

In [ ]:
client.messages.parse(model=MODEL, max_tokens=300, system=CODEBOOK, messages=[{"role": "user", "content": SENTENCES.text[3]}], output_format=Annotation, thinking={"type": "disabled"}).parsed_output

## 4. Everything else is a loop

In [ ]:
SENTENCES["claude"] = [label(t) for t in SENTENCES.text]

In [ ]:
SENTENCES[["text", "my_label", "claude"]]

In [ ]:
(SENTENCES.my_label == SENTENCES.claude).mean()

Where we disagree: who is right?

In [ ]:
SENTENCES[SENTENCES.my_label != SENTENCES.claude][["text", "my_label", "claude"]]

The same sentence, five times. Does the answer change?

In [ ]:
pd.Series([label(SENTENCES.text[12]) for _ in range(5)]).value_counts()

## 5. A batch: all the requests in one go, at half price

For a big corpus you do not call one by one. You send a **list of requests**, each with an identifier (`custom_id`), and fetch the answers later. The price is 50% lower. Most batches finish within an hour (24 hours at most), up to 100,000 requests. Start it early and come back to it: **do not wait for it live.**

In [ ]:
REQUESTS = [{"custom_id": f"s{i}", "params": params(t)} for i, t in enumerate(SENTENCES.text)]

In [ ]:
len(REQUESTS), REQUESTS[0]["custom_id"]

In [ ]:
REQUESTS[0]

The format of a batch is JSON Lines: one request per line. This is the file you archive.

In [ ]:
open("requests.jsonl", "w").write("\n".join(json.dumps(x, ensure_ascii=False) for x in REQUESTS))

In [ ]:
print(open("requests.jsonl").read()[:700])

In [ ]:
batch = client.messages.batches.create(requests=REQUESTS)

In [ ]:
batch.id, batch.processing_status

Check on it. Run this cell again until `processing_status` is `ended`.

In [ ]:
batch = client.messages.batches.retrieve(batch.id); batch.processing_status, batch.request_counts

The results come back in any order: match them to the sentences with `custom_id`.

In [ ]:
RESULTS = {x.custom_id: x.result for x in client.messages.batches.results(batch.id)}

In [ ]:
{k: v.type for k, v in list(RESULTS.items())[:5]}

In [ ]:
SENTENCES["batch"] = [json.loads(text_of(RESULTS[f"s{i}"].message))["domain"] if RESULTS[f"s{i}"].type == "succeeded" else None for i in range(len(SENTENCES))]

In [ ]:
(SENTENCES.claude == SENTENCES.batch).mean()

Same codebook, same model: the batch and the one-by-one calls should agree on almost everything. Where they do not, the answer is not deterministic. That is why you report the number of runs.

## 6. Keep what a referee would ask for

In [ ]:
SENTENCES.to_csv("labels.csv", index=False)

In [ ]:
json.dump({"model": MODEL, "date": str(date.today()), "codebook_sha256": hashlib.sha256(CODEBOOK.encode()).hexdigest(), "schema": SCHEMA, "thinking": "disabled", "batch_id": batch.id}, open("params.json", "w"), indent=2)

In [ ]:
print(open("params.json").read())

**What you just saw:** an API call is an HTTP request; a label is a field in a JSON answer; annotating a corpus is a loop, or a batch; and what makes it science is not the code but the codebook, the validation against a human coder, and the file `params.json`.